In [1]:
import sys, os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["PYTORCH_USE_CUDA_DSA"] = "1"
import numpy as np
import torch
import yaml
import pickle as pk
# autoreload modules
%load_ext autoreload
%autoreload 2

import matplotlib
%matplotlib inline
import matplotlib.pyplot as pl
import numpy as np
import sys,os
import readgadget
import MAS_library as MASL
import pickle as pk
import readfof
import matplotlib

import matplotlib.pyplot as pl
pl.rc('text', usetex=True)
# Palatino
pl.rc('font', family='DejaVu Sans')
%matplotlib inline

import readfof
import sys, os
import numpy as np
import pickle as pk 
# from nbodykit.lab import *
import h5py as h5
import numpy as np
import Pk_library as PKL
import MAS_library as MASL
import yaml
# import galactic_wavelets as gw

from train_dtai_wvel import *


# pl.rcParams['figure.dpi'] = 250

%load_ext Cython



        



In [2]:
dev = torch.device("cuda")
nrand_subsel = 128
# subsel_types = ['no_vel']
subsel_types = ['all']
# subsel_type = 'all'
# subsel_type = 'no_highz_no_vel'
for subsel_type in subsel_types:
    print(subsel_type)
    Mstar_cut = 9.0
    add_space_token = False
    # add_space_token = True
    
    n_evals = 8
    
    # checkpoint = torch.load(f'/projects/bdne/spandey3/GOTHAM/model_checkpoints/camels_photo_velx/model_hres_encdec_ddp_PM_nvocab_64_nembed_128_nhead_8_nrandsubsel_64_subselDMOfields_{subsel_type}_Mstarcut_{Mstar_cut}_spacetoken_{add_space_token}.pt')
    # checkpoint = torch.load(f'/projects/bdne/spandey3/GOTHAM/model_checkpoints/camels_photo_velx/model_hres_encdec_ddp_PM_nvocab_64_nembed_256_nhead_8_nrandsubsel_{nrand_subsel}_subselDMOfields_{subsel_type}_Mstarcut_{Mstar_cut}_spacetoken_{add_space_token}.pt')    
    # checkpoint = torch.load(f'/projects/bdne/spandey3/GOTHAM/model_checkpoints/camels_photo_velx/all_run3.pt')
    checkpoint = torch.load('/projects/bdne/spandey3/GOTHAM/model_checkpoints/camels_photo_velx/TEST2_model_hres_encdec_ddp_grid_32_nvocab_64_nembed_256_nhead_8_nrandsubsel_128_subselDMOfields_all_Mstarcut_9.0_spacetoken_False_maxiter_1500_lr_0.0003.pt')
    
    HaloConfig = checkpoint['config']
    print(checkpoint['best_val_loss'])
    learning_rate = 5e-4
    model = HaloDecoderModel(HaloConfig).to(dev)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    
    model.load_state_dict(checkpoint['model'])
    model.eval()
    print()
    model = model.bfloat16()
    
    params_all = np.loadtxt('/projects/bdne/spandey3/GOTHAM/prep_data/camels_tng_LH_params.txt', usecols=range(1, 7))
    
    
    
    
    nvocab = 64
    grid = 8
    # add_space_token = False
    BoxSize = 25.
    xall = (np.linspace(0, BoxSize/grid, nvocab + 1))
    xarray = 0.5 * (xall[1:] + xall[:-1])
    yarray = np.copy(xarray)
    zarray = np.copy(xarray)
    
    from tqdm import tqdm
    saved_all = {}
    # isim_fid_array = np.arange(975, 990)
    # isim_fid_array = np.arange(990, 1000)    
    import numpy as np
    arr = np.arange(1000)
    split_arr = np.array_split(arr, 8)
    isim_fid_array = np.concatenate([part[int(0.925 * len(part)):] for part in split_arr])

    for isim_fid in tqdm(isim_fid_array):
        ldir = '/work/hdd/bdne/spandey3/camels_tng/gotham_data/LH/'
        savefname_dmo_fields = f'{ldir}/DMO_fields/DMO_fields_grid_32_isim_{isim_fid}_nrandsubsel_512_MAS_NGP_nsnaps_5.pkl'
        # df = pk.load(open(f'{sdir}/subhalo_density3Dgrid_32_isim_{isim_fid}_nrandsubsel_512_nvocab64_wSDSS_photometry_vel_hres.pkl','rb'))    
        # delta_box_all_squeezed_0 = df['delta_box_all_squeezed']
        df = pk.load(open(savefname_dmo_fields,'rb'))
        dmo_fields_all = torch.moveaxis(torch.tensor(df['dmo_fields_all']).to(torch.float16), -1, 1)
    
        # savefname_gals = f'{ldir}/gal_props/galaxy_props_snap_90_grid_{nvocab}_isim_{isim_fid}_nrandsubsel_512_nvocab{nvocab}_wSDSS_photometry_velx.pkl'
        savefname_gals = f'{ldir}/gal_props/galaxy_props_snap_{90}_grid_{nvocab}_isim_{isim_fid}_nrandsubsel_{512}_nvocab{nvocab}_spacetoken_{add_space_token}_wSDSS_photometry_velx_Mstarcut_{Mstar_cut}.pkl'    
        df = pk.load(open(savefname_gals,'rb'))
        dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_0 = df['story_full']
    
        param_0 = params_all[isim_fid][None, :]
        param_0 = np.repeat(param_0, len(dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_0), axis=0)
    
        n1 = 8**3
        test_data_halos = dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_0[:n1]
    
        test_data_dm = dmo_fields_all[:n1]
        params_test = param_0[:n1]
    
        x = torch.tensor(test_data_halos[:, :-1])
        y = torch.tensor(test_data_halos[:, 1:])
        dm = torch.tensor(test_data_dm)
        mask_test_orig = x != 1
        mask_test = torch.logical_not(mask_test_orig)
        masked_logits = torch.zeros(mask_test.shape)
        mask_test_final = masked_logits.masked_fill(mask_test, float('-inf'))
        mask_test = mask_test_final[:,None,:]
        x, y = torch.tensor(x), torch.tensor(y)
        x_test = x.long()
        y_test = y.long()
        dm_test = dm.bfloat16()
        mask_test = torch.tensor(mask_test).bfloat16()
        params_test = torch.tensor(params_test).bfloat16()
    
        x_test_gpu = x_test.to(dev)
        y_test_gpu = y_test.to(dev)
        dm_test_gpu = dm_test.to(dev)
        mask_test_gpu = mask_test.to(dev)
        params_test_gpu = params_test.to(dev)
    
        if subsel_type == 'no_highz':
            indices = torch.arange(6)
        elif subsel_type == 'no_highz_no_vel':
            indices = torch.arange(3)
        elif subsel_type == 'no_highz_no_env':
            indices = torch.from_numpy(np.array([0,3,4,5]))        
        elif subsel_type == 'no_vel':
            indices = torch.cat([torch.arange(i, i + 3) for i in range(0, 30, 6)])
        elif subsel_type == 'no_env':        
            indices1 = torch.cat([torch.arange(i+3, i + 6) for i in range(0, 30, 6)])
            indices2 = torch.cat([torch.arange(i, i + 1) for i in range(0, 30, 6)])
            indices, _ = torch.sort(torch.cat([indices1, indices2]))
        else:
            indices = torch.arange(dm_test_gpu.shape[1])
    
        dm_test_gpu = dm_test_gpu[:,indices,...]
        
    
        def get_batch(split, ji=0, batch_size=None):
            if split == 'test':
                x = x_test_gpu
                y = y_test_gpu
                mask = mask_test_gpu
                dm = dm_test_gpu       
                params = params_test_gpu 
    
            if batch_size is not None:
                x = x[batch_size*(ji):batch_size*(ji+1)].to(dev, non_blocking=True)
                y = y[batch_size*(ji):batch_size*(ji+1)].to(dev, non_blocking=True)
                mask = mask[batch_size*(ji):batch_size*(ji+1)].to(dev, non_blocking=True)
                dm = dm[batch_size*(ji):batch_size*(ji+1)].to(dev, non_blocking=True)
                params = params[batch_size*(ji):batch_size*(ji+1)].to(dev, non_blocking=True)
    
            return x, y, mask, dm, params
    
        batch_size = 8**3
        X_val, Y_val, MASK_val, DM_val, PARAMS = get_batch('test', 0, batch_size)
        # DM_val = torch.moveaxis(DM_val, -1, 1)
    
        max_new_tokens = df['max_sentence_length']
        nvocab_tot = df['end_token'] + 1
    
        bins_digitize = np.linspace(-1e-3, 1, nvocab)
    
        prop_min = df['prop_min']
        prop_max = df['prop_max']
        start_token = df['start_token']
        pad_token = df['pad_token']
        end_token =  df['end_token']
        space_token = df['space_token']
        
        dim_pos = 3
        dim_prop = len(prop_min)   
        dim_tot = dim_pos + dim_prop
        
    
        BoxSize = 25.
        grid = 8
        xmin = BoxSize/grid/2
    
        MAS     = 'NGP'  #mass-assigment scheme
        
        def get_prop_pos(X_val):
            pos_infer_all = []
            prop_infer_all = []
            
            for jx in range(grid):
                for jy in range(grid):
                    for jz in range(grid):
                        sentence_here = X_val[jx, jy, jz]
                        if add_space_token:
                            ntokens_per_halo = dim_tot + 1
                        else:
                            ntokens_per_halo = dim_tot
                        ind_start_token = np.where(sentence_here == start_token)[0][0]
                        if end_token in sentence_here:
                            ind_end_token = np.where(sentence_here == end_token)[0]
                            try:
                                Nhalos_here = ((ind_end_token - ind_start_token - 1) / ntokens_per_halo)[0]
                            except:
                                print(ind_end_token, ind_start_token, ntokens_per_halo)
                            if (int(Nhalos_here) - Nhalos_here) != 0:
                                Nhalos_here = ((ind_end_token - ind_start_token ) / ntokens_per_halo)[0]
                                # print(Nhalos_here, Nhalos_here2, 'Nhalos_here is not an integer')
                                # pass
                            if (int(Nhalos_here) - Nhalos_here) != 0:
                                # print(Nhalos_here, 'Nhalos_here is not an integer')
                                pass
                            else:
                                if Nhalos_here > 0:
                                    for jh in range(int(Nhalos_here)):
                                        try:
                                            coord_x = (xarray[sentence_here[ind_start_token + jh*ntokens_per_halo + 1]] + (BoxSize/grid)*jx) % BoxSize
                                            coord_y = (yarray[sentence_here[ind_start_token + jh*ntokens_per_halo + 2]] + (BoxSize/grid)*jy) % BoxSize
                                            coord_z = (zarray[sentence_here[ind_start_token + jh*ntokens_per_halo + 3]] + (BoxSize/grid)*jz) % BoxSize
                                            pos_infer_all.append([coord_x, coord_y, coord_z])
                                            prop_all = np.zeros(dim_prop)        
                                            for jp in range(dim_prop):
                                                # noise_val = np.random.uniform(-delta_bin/2, delta_bin/2)
                                                noise_val = 0
                                                bin_val_jp = sentence_here[ind_start_token + jh*ntokens_per_halo + 4 + jp]
                                                try:
                                                    prop_all[jp] = prop_min[jp] + (prop_max[jp] - prop_min[jp]) * (bins_digitize[bin_val_jp] + noise_val)
                                                except:
                                                    pass
                                                    # print(sentence_here)
                                            prop_infer_all.append(prop_all)
         
                                        except:
                                            pass
                                            # print(sentence_here[ind_start_token], ntokens_per_halo)
                        else:           
                            # print('End token not found')
                            pass
            
            pos_infer_all = np.array(pos_infer_all)
            prop_infer_all = np.array(prop_infer_all)
            return pos_infer_all, prop_infer_all
        X_val_rs = np.reshape(dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_0, (grid, grid, grid, dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_0.shape[-1])).astype(int)
        pos_truth_all, prop_truth_all = get_prop_pos(X_val_rs)
        
    
    
    
        from tqdm import tqdm
    
        fac = 1
        nvox_samp = batch_size
        nbatches = fac
        nvox_per_batch = nvox_samp // nbatches
    
        pos_infer_all_evals, prop_infer_all_evals = {}, {}
        for jev in range(n_evals):
            idx_all = np.ones((nvox_samp, max_new_tokens))
            
            for jb in (range(nbatches)):
        
                idx_inp = torch.zeros((nvox_per_batch, 1), dtype=torch.long, device=dev)
        
                DM_val_jb = DM_val[jb*nvox_per_batch:(jb+1)*nvox_per_batch,...]
        
                param_jb = PARAMS[jb*nvox_per_batch:(jb+1)*nvox_per_batch,...]
        
                new_samples_jb = np.ones((nvox_per_batch, max_new_tokens))
        
                ind_jb = np.arange(nvox_per_batch)
        
                new_samples_jb[:, 0] = idx_inp[:,0].cpu().detach().numpy() + start_token
                # idx = idx_inp
                for jt in range(1, max_new_tokens):
                    if len(ind_jb) > 0:
                        # crop idx to the last block_size tokens
                        idx_cond = torch.tensor(new_samples_jb[ind_jb, :jt], dtype=torch.long, device=dev)
                        # get the predictions
                        logits, loss = model(idx_cond, DM_val_jb[ind_jb,...], params=param_jb[ind_jb,...])
                        # focus only on the last time step
                        logits = logits[:, -1, :] # becomes (B, C)
                        # apply softmax to get probabilities
                        probs = F.softmax(logits, dim=-1) # (B, C)
                        
                        # sample from the distribution
                        idx_next = torch.multinomial(probs, num_samples=1).cpu().detach().numpy() # (B, 1)
                        # import pdb; pdb.set_trace()
                        # append sampled index to the running sequence
                        # if idx_next == end_token:
                            # break
                        new_samples_jb[ind_jb, jt] = idx_next[:,0]
                        ind_to_del = np.where(idx_next[:,0] == end_token)[0]
                        # for jv in range(len(ind_jb)):
                            # if idx_next[jv, 0] == end_token:
                        if len(ind_to_del) > 0:
                            ind_jb = np.delete(ind_jb, ind_to_del)
        
                idx_all[jb*nvox_per_batch:(jb+1)*nvox_per_batch, :] = new_samples_jb
        
            idx_all_rs = np.reshape(np.array(idx_all), (grid, grid, grid, idx_all.shape[-1])).astype(int)
            X_val_rs = np.reshape(dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_0, (grid, grid, grid, dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_0.shape[-1])).astype(int)
            pos_infer_all, prop_infer_all = get_prop_pos(idx_all_rs)
            pos_infer_all_evals[jev] = pos_infer_all
            prop_infer_all_evals[jev] = prop_infer_all
        
        saved_all[isim_fid] = {'pos_truth':pos_truth_all, 'prop_truth':prop_truth_all, 'pos_infer_all_evals':pos_infer_all_evals, 'prop_infer_all_evals':prop_infer_all_evals}
    
        # pk.dump(saved_all, open(f'/projects/bdne/spandey3/GOTHAM/data/infer_cats/RUN2_pos_prop_infer_all_nevals_{n_evals}_{subsel_type}_Mstarcut_{Mstar_cut}_spacetoken_{add_space_token}_nrandsubsel_{nrand_subsel}.pk','wb')) 
        pk.dump(saved_all, open(f'/projects/bdne/spandey3/GOTHAM/data/infer_cats/FINAL_pos_prop_infer_all_nevals_{n_evals}_{subsel_type}_Mstarcut_{Mstar_cut}_spacetoken_{add_space_token}_nrandsubsel_{nrand_subsel}.pk','wb')) 
    
    




all


/tmp/ipykernel_3966956/3450884493.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('/projects/bdne/spandey3/GOTHAM/model_checkpoints/camels_photo

tensor(2.1420)
Using flash:  False
Using flash:  False
Using flash:  False
Using flash:  False
Using flash:  False
Using flash:  False
Using flash:  False
Using flash:  False
number of parameters: 15.16M



  0%|          | 0/80 [00:00<?, ?it/s]/tmp/ipykernel_3966956/3450884493.py:78: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  dm = torch.tensor(test_data_dm)
/tmp/ipykernel_3966956/3450884493.py:84: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x, y = torch.tensor(x), torch.tensor(y)
/tmp/ipykernel_3966956/3450884493.py:88: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask_test = torch.tensor(mask_test).bfloat16()
  0%|          | 0/80 [00:05<?, ?it/s]


KeyboardInterrupt: 

In [2]:
dev = torch.device("cuda")
nrand_subsel = 128
# subsel_types = ['no_env', 'no_highz']
subsel_types = ['no_highz']
# subsel_types = ['all']
# subsel_type = 'all'
# subsel_type = 'no_highz_no_vel'
for subsel_type in subsel_types:
    print(subsel_type)
    Mstar_cut = 9.0
    add_space_token = False
    # add_space_token = True
    
    n_evals = 8
    
    # checkpoint = torch.load(f'/projects/bdne/spandey3/GOTHAM/model_checkpoints/camels_photo_velx/model_hres_encdec_ddp_PM_nvocab_64_nembed_128_nhead_8_nrandsubsel_64_subselDMOfields_{subsel_type}_Mstarcut_{Mstar_cut}_spacetoken_{add_space_token}.pt')
    checkpoint = torch.load(f'/projects/bdne/spandey3/GOTHAM/model_checkpoints/camels_photo_velx/model_hres_encdec_ddp_PM_nvocab_64_nembed_256_nhead_8_nrandsubsel_{nrand_subsel}_subselDMOfields_{subsel_type}_Mstarcut_{Mstar_cut}_spacetoken_{add_space_token}.pt')    

    HaloConfig = checkpoint['config']
    print(checkpoint['best_val_loss'])
    learning_rate = 5e-4
    model = HaloDecoderModel(HaloConfig).to(dev)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    
    model.load_state_dict(checkpoint['model'])
    model.eval()
    print()
    model = model.bfloat16()
    
    params_all = np.loadtxt('/projects/bdne/spandey3/GOTHAM/prep_data/camels_tng_LH_params.txt', usecols=range(1, 7))
    
    
    
    
    nvocab = 64
    grid = 8
    # add_space_token = False
    BoxSize = 25.
    xall = (np.linspace(0, BoxSize/grid, nvocab + 1))
    xarray = 0.5 * (xall[1:] + xall[:-1])
    yarray = np.copy(xarray)
    zarray = np.copy(xarray)
    
    from tqdm import tqdm
    saved_all = {}
    # isim_fid_array = np.arange(975, 990)
    # isim_fid_array = np.arange(990, 1000)    
    import numpy as np
    arr = np.arange(1000)
    split_arr = np.array_split(arr, 8)
    isim_fid_array = np.concatenate([part[int(0.925 * len(part)):] for part in split_arr])

    for isim_fid in tqdm(isim_fid_array):
        ldir = '/work/hdd/bdne/spandey3/camels_tng/gotham_data/LH/'
        savefname_dmo_fields = f'{ldir}/DMO_fields/DMO_fields_grid_32_isim_{isim_fid}_nrandsubsel_512_MAS_NGP_nsnaps_5.pkl'
        # df = pk.load(open(f'{sdir}/subhalo_density3Dgrid_32_isim_{isim_fid}_nrandsubsel_512_nvocab64_wSDSS_photometry_vel_hres.pkl','rb'))    
        # delta_box_all_squeezed_0 = df['delta_box_all_squeezed']
        df = pk.load(open(savefname_dmo_fields,'rb'))
        dmo_fields_all = torch.moveaxis(torch.tensor(df['dmo_fields_all']).to(torch.float16), -1, 1)
    
        # savefname_gals = f'{ldir}/gal_props/galaxy_props_snap_90_grid_{nvocab}_isim_{isim_fid}_nrandsubsel_512_nvocab{nvocab}_wSDSS_photometry_velx.pkl'
        savefname_gals = f'{ldir}/gal_props/galaxy_props_snap_{90}_grid_{nvocab}_isim_{isim_fid}_nrandsubsel_{512}_nvocab{nvocab}_spacetoken_{add_space_token}_wSDSS_photometry_velx_Mstarcut_{Mstar_cut}.pkl'    
        df = pk.load(open(savefname_gals,'rb'))
        dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_0 = df['story_full']
    
        param_0 = params_all[isim_fid][None, :]
        param_0 = np.repeat(param_0, len(dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_0), axis=0)
    
        n1 = 8**3
        test_data_halos = dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_0[:n1]
    
        test_data_dm = dmo_fields_all[:n1]
        params_test = param_0[:n1]
    
        x = torch.tensor(test_data_halos[:, :-1])
        y = torch.tensor(test_data_halos[:, 1:])
        dm = torch.tensor(test_data_dm)
        mask_test_orig = x != 1
        mask_test = torch.logical_not(mask_test_orig)
        masked_logits = torch.zeros(mask_test.shape)
        mask_test_final = masked_logits.masked_fill(mask_test, float('-inf'))
        mask_test = mask_test_final[:,None,:]
        x, y = torch.tensor(x), torch.tensor(y)
        x_test = x.long()
        y_test = y.long()
        dm_test = dm.bfloat16()
        mask_test = torch.tensor(mask_test).bfloat16()
        params_test = torch.tensor(params_test).bfloat16()
    
        x_test_gpu = x_test.to(dev)
        y_test_gpu = y_test.to(dev)
        dm_test_gpu = dm_test.to(dev)
        mask_test_gpu = mask_test.to(dev)
        params_test_gpu = params_test.to(dev)
    
        if subsel_type == 'no_highz':
            indices = torch.arange(6)
        elif subsel_type == 'no_highz_no_vel':
            indices = torch.arange(3)
        elif subsel_type == 'no_highz_no_env':
            indices = torch.from_numpy(np.array([0,3,4,5]))        
        elif subsel_type == 'no_vel':
            indices = torch.cat([torch.arange(i, i + 3) for i in range(0, 30, 6)])
        elif subsel_type == 'no_env':        
            indices1 = torch.cat([torch.arange(i+3, i + 6) for i in range(0, 30, 6)])
            indices2 = torch.cat([torch.arange(i, i + 1) for i in range(0, 30, 6)])
            indices, _ = torch.sort(torch.cat([indices1, indices2]))
        else:
            indices = torch.arange(dm_test_gpu.shape[1])
    
        dm_test_gpu = dm_test_gpu[:,indices,...]
        
    
        def get_batch(split, ji=0, batch_size=None):
            if split == 'test':
                x = x_test_gpu
                y = y_test_gpu
                mask = mask_test_gpu
                dm = dm_test_gpu       
                params = params_test_gpu 
    
            if batch_size is not None:
                x = x[batch_size*(ji):batch_size*(ji+1)].to(dev, non_blocking=True)
                y = y[batch_size*(ji):batch_size*(ji+1)].to(dev, non_blocking=True)
                mask = mask[batch_size*(ji):batch_size*(ji+1)].to(dev, non_blocking=True)
                dm = dm[batch_size*(ji):batch_size*(ji+1)].to(dev, non_blocking=True)
                params = params[batch_size*(ji):batch_size*(ji+1)].to(dev, non_blocking=True)
    
            return x, y, mask, dm, params
    
        batch_size = 8**3
        X_val, Y_val, MASK_val, DM_val, PARAMS = get_batch('test', 0, batch_size)
        # DM_val = torch.moveaxis(DM_val, -1, 1)
    
        max_new_tokens = df['max_sentence_length']
        nvocab_tot = df['end_token'] + 1
    
        bins_digitize = np.linspace(-1e-3, 1, nvocab)
    
        prop_min = df['prop_min']
        prop_max = df['prop_max']
        start_token = df['start_token']
        pad_token = df['pad_token']
        end_token =  df['end_token']
        space_token = df['space_token']
        
        dim_pos = 3
        dim_prop = len(prop_min)   
        dim_tot = dim_pos + dim_prop
        
    
        BoxSize = 25.
        grid = 8
        xmin = BoxSize/grid/2
    
        MAS     = 'NGP'  #mass-assigment scheme
        
        def get_prop_pos(X_val):
            pos_infer_all = []
            prop_infer_all = []
            
            for jx in range(grid):
                for jy in range(grid):
                    for jz in range(grid):
                        sentence_here = X_val[jx, jy, jz]
                        if add_space_token:
                            ntokens_per_halo = dim_tot + 1
                        else:
                            ntokens_per_halo = dim_tot
                        ind_start_token = np.where(sentence_here == start_token)[0][0]
                        if end_token in sentence_here:
                            ind_end_token = np.where(sentence_here == end_token)[0]
                            try:
                                Nhalos_here = ((ind_end_token - ind_start_token - 1) / ntokens_per_halo)[0]
                            except:
                                print(ind_end_token, ind_start_token, ntokens_per_halo)
                            if (int(Nhalos_here) - Nhalos_here) != 0:
                                Nhalos_here = ((ind_end_token - ind_start_token ) / ntokens_per_halo)[0]
                                # print(Nhalos_here, Nhalos_here2, 'Nhalos_here is not an integer')
                                # pass
                            if (int(Nhalos_here) - Nhalos_here) != 0:
                                # print(Nhalos_here, 'Nhalos_here is not an integer')
                                pass
                            else:
                                if Nhalos_here > 0:
                                    for jh in range(int(Nhalos_here)):
                                        try:
                                            coord_x = (xarray[sentence_here[ind_start_token + jh*ntokens_per_halo + 1]] + (BoxSize/grid)*jx) % BoxSize
                                            coord_y = (yarray[sentence_here[ind_start_token + jh*ntokens_per_halo + 2]] + (BoxSize/grid)*jy) % BoxSize
                                            coord_z = (zarray[sentence_here[ind_start_token + jh*ntokens_per_halo + 3]] + (BoxSize/grid)*jz) % BoxSize
                                            pos_infer_all.append([coord_x, coord_y, coord_z])
                                            prop_all = np.zeros(dim_prop)        
                                            for jp in range(dim_prop):
                                                # noise_val = np.random.uniform(-delta_bin/2, delta_bin/2)
                                                noise_val = 0
                                                bin_val_jp = sentence_here[ind_start_token + jh*ntokens_per_halo + 4 + jp]
                                                try:
                                                    prop_all[jp] = prop_min[jp] + (prop_max[jp] - prop_min[jp]) * (bins_digitize[bin_val_jp] + noise_val)
                                                except:
                                                    pass
                                                    # print(sentence_here)
                                            prop_infer_all.append(prop_all)
         
                                        except:
                                            pass
                                            # print(sentence_here[ind_start_token], ntokens_per_halo)
                        else:           
                            # print('End token not found')
                            pass
            
            pos_infer_all = np.array(pos_infer_all)
            prop_infer_all = np.array(prop_infer_all)
            return pos_infer_all, prop_infer_all
        X_val_rs = np.reshape(dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_0, (grid, grid, grid, dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_0.shape[-1])).astype(int)
        pos_truth_all, prop_truth_all = get_prop_pos(X_val_rs)
        
    
    
    
        from tqdm import tqdm
    
        fac = 1
        nvox_samp = batch_size
        nbatches = fac
        nvox_per_batch = nvox_samp // nbatches
    
        pos_infer_all_evals, prop_infer_all_evals = {}, {}
        for jev in range(n_evals):
            idx_all = np.ones((nvox_samp, max_new_tokens))
            
            for jb in (range(nbatches)):
        
                idx_inp = torch.zeros((nvox_per_batch, 1), dtype=torch.long, device=dev)
        
                DM_val_jb = DM_val[jb*nvox_per_batch:(jb+1)*nvox_per_batch,...]
        
                param_jb = PARAMS[jb*nvox_per_batch:(jb+1)*nvox_per_batch,...]
        
                new_samples_jb = np.ones((nvox_per_batch, max_new_tokens))
        
                ind_jb = np.arange(nvox_per_batch)
        
                new_samples_jb[:, 0] = idx_inp[:,0].cpu().detach().numpy() + start_token
                # idx = idx_inp
                for jt in range(1, max_new_tokens):
                    if len(ind_jb) > 0:
                        # crop idx to the last block_size tokens
                        idx_cond = torch.tensor(new_samples_jb[ind_jb, :jt], dtype=torch.long, device=dev)
                        # get the predictions
                        logits, loss = model(idx_cond, DM_val_jb[ind_jb,...], params=param_jb[ind_jb,...])
                        # focus only on the last time step
                        logits = logits[:, -1, :] # becomes (B, C)
                        # apply softmax to get probabilities
                        probs = F.softmax(logits, dim=-1) # (B, C)
                        # sample from the distribution
                        idx_next = torch.multinomial(probs, num_samples=1).cpu().detach().numpy() # (B, 1)
        
                        # append sampled index to the running sequence
                        # if idx_next == end_token:
                            # break
                        new_samples_jb[ind_jb, jt] = idx_next[:,0]
                        ind_to_del = np.where(idx_next[:,0] == end_token)[0]
                        # for jv in range(len(ind_jb)):
                            # if idx_next[jv, 0] == end_token:
                        if len(ind_to_del) > 0:
                            ind_jb = np.delete(ind_jb, ind_to_del)
        
                idx_all[jb*nvox_per_batch:(jb+1)*nvox_per_batch, :] = new_samples_jb
        
            idx_all_rs = np.reshape(np.array(idx_all), (grid, grid, grid, idx_all.shape[-1])).astype(int)
            X_val_rs = np.reshape(dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_0, (grid, grid, grid, dfhalo_ngp_xyzM_tokenized_padded_ended_squeezed_0.shape[-1])).astype(int)
            pos_infer_all, prop_infer_all = get_prop_pos(idx_all_rs)
            pos_infer_all_evals[jev] = pos_infer_all
            prop_infer_all_evals[jev] = prop_infer_all
        
        saved_all[isim_fid] = {'pos_truth':pos_truth_all, 'prop_truth':prop_truth_all, 'pos_infer_all_evals':pos_infer_all_evals, 'prop_infer_all_evals':prop_infer_all_evals}
    
    pk.dump(saved_all, open(f'/projects/bdne/spandey3/GOTHAM/data/infer_cats/pos_prop_infer_all_nevals_{n_evals}_{subsel_type}_Mstarcut_{Mstar_cut}_spacetoken_{add_space_token}_nrandsubsel_{nrand_subsel}.pk','wb')) 
    
    






no_highz


/tmp/ipykernel_1118897/1760258459.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(f'/projects/bdne/spandey3/GOTHAM/model_checkpoints/camels_phot

tensor(2.2427)
Using flash:  False
Using flash:  False
Using flash:  False
Using flash:  False
Using flash:  False
Using flash:  False
Using flash:  False
Using flash:  False
number of parameters: 15.12M



  0%|          | 0/80 [00:00<?, ?it/s]/tmp/ipykernel_1118897/1760258459.py:77: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  dm = torch.tensor(test_data_dm)
/tmp/ipykernel_1118897/1760258459.py:83: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x, y = torch.tensor(x), torch.tensor(y)
/tmp/ipykernel_1118897/1760258459.py:87: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask_test = torch.tensor(mask_test).bfloat16()
100%|██████████| 80/80 [50:23<00:00, 37.80s/it]
